In [ ]:
import os, sys, subprocess

WORK = "/kaggle/working"
REPO = f"{WORK}/clipping-studio"
STUDIO_PY = f"{WORK}/venv-studio/bin/python"
VLLM_PY = f"{WORK}/venv-vllm/bin/python"
LLM_PRESET = globals().get("LLM_PRESET", "qwen3-4b-awq")   # or "gemma4-e2b"


def sh(cmd, check=True, cwd=None, env=None):
    """Run a shell command, stream its output live, stop cleanly when the cell is interrupted."""
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, executable="/bin/bash",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in p.stdout:
            print(line, end="")
        p.wait()
    except KeyboardInterrupt:
        p.terminate()
        raise
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p.returncode

# ============ CELL 1 - Get the code ============
REPO_URL = "https://github.com/YOUR_GITHUB_USER/youtube-clipping-studio.git"  # <-- set to your repo
LOCAL_DATASET = "/kaggle/input/clipping-studio"   # alternative: upload this project as a Kaggle Dataset
LLM_PRESET = "qwen3-4b-awq"                       # "qwen3-4b-awq" (default) or "gemma4-e2b"

import shutil
if os.path.isdir(f"{REPO}/backend"):
    print("Repo already present - updating")
    if os.path.isdir(f"{REPO}/.git"):
        sh("git pull --ff-only", check=False, cwd=REPO)
elif os.path.isdir(LOCAL_DATASET):
    print("Copying project from Kaggle dataset")
    shutil.copytree(LOCAL_DATASET, REPO)
else:
    sh(f"git clone --depth 1 {REPO_URL} {REPO}")
print("Code ready:", os.listdir(REPO))


In [ ]:
import os, sys, subprocess

WORK = "/kaggle/working"
REPO = f"{WORK}/clipping-studio"
STUDIO_PY = f"{WORK}/venv-studio/bin/python"
VLLM_PY = f"{WORK}/venv-vllm/bin/python"
LLM_PRESET = globals().get("LLM_PRESET", "qwen3-4b-awq")   # or "gemma4-e2b"


def sh(cmd, check=True, cwd=None, env=None):
    """Run a shell command, stream its output live, stop cleanly when the cell is interrupted."""
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, executable="/bin/bash",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in p.stdout:
            print(line, end="")
        p.wait()
    except KeyboardInterrupt:
        p.terminate()
        raise
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p.returncode

# ============ CELL 2 - Python 3.12 virtual environments (separate from the Kaggle kernel) ============
sh(f"{sys.executable} -m pip install -q uv")
UV = f"{sys.executable} -m uv"
sh(f"{UV} venv --python 3.12 {WORK}/venv-studio")   # FastAPI + faster-whisper
sh(f"{UV} venv --python 3.12 {WORK}/venv-vllm")     # vLLM (own torch/CUDA stack)
sh(f"{STUDIO_PY} --version && {VLLM_PY} --version")


In [ ]:
import os, sys, subprocess

WORK = "/kaggle/working"
REPO = f"{WORK}/clipping-studio"
STUDIO_PY = f"{WORK}/venv-studio/bin/python"
VLLM_PY = f"{WORK}/venv-vllm/bin/python"
LLM_PRESET = globals().get("LLM_PRESET", "qwen3-4b-awq")   # or "gemma4-e2b"


def sh(cmd, check=True, cwd=None, env=None):
    """Run a shell command, stream its output live, stop cleanly when the cell is interrupted."""
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, executable="/bin/bash",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in p.stdout:
            print(line, end="")
        p.wait()
    except KeyboardInterrupt:
        p.terminate()
        raise
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p.returncode

# ============ CELL 3 - Install FFmpeg + dependencies ============
UV = f"{sys.executable} -m uv"
sh("apt-get update -qq && apt-get install -y -qq ffmpeg fontconfig fonts-dejavu-core fonts-liberation > /dev/null")
sh("ffmpeg -version | head -n 1")
sh("fc-cache -f > /dev/null 2>&1", check=False)

# Studio venv (backend, yt-dlp, faster-whisper, CUDA libs)
sh(f"{UV} pip install --python {STUDIO_PY} -r {REPO}/requirements.txt")
sh(f"{UV} pip install --python {STUDIO_PY} -U 'yt-dlp[default]'", check=False)  # YouTube changes often - stay fresh
# Optional JS runtime that recent yt-dlp versions use for YouTube
sh("curl -fsSL https://deno.land/install.sh | sh -s -- -y > /dev/null 2>&1", check=False)

# vLLM venv (large download, a few minutes)
sh(f"{UV} pip install --python {VLLM_PY} -r {REPO}/requirements-vllm.txt")
if LLM_PRESET == "gemma4-e2b":
    sh(f"{UV} pip install --python {VLLM_PY} -U 'vllm>=0.29'")
print("\nAll dependencies installed.")


In [ ]:
import os, sys, subprocess

WORK = "/kaggle/working"
REPO = f"{WORK}/clipping-studio"
STUDIO_PY = f"{WORK}/venv-studio/bin/python"
VLLM_PY = f"{WORK}/venv-vllm/bin/python"
LLM_PRESET = globals().get("LLM_PRESET", "qwen3-4b-awq")   # or "gemma4-e2b"


def sh(cmd, check=True, cwd=None, env=None):
    """Run a shell command, stream its output live, stop cleanly when the cell is interrupted."""
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, executable="/bin/bash",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in p.stdout:
            print(line, end="")
        p.wait()
    except KeyboardInterrupt:
        p.terminate()
        raise
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p.returncode

# ============ CELL 4 - Start the LOCAL vLLM server (port 8001, GPU 0) ============
# Runs in the background and survives after this cell finishes. If it fails (OOM etc.) the studio
# still works with its heuristic engine - so this cell never raises.
sh(f"{sys.executable} scripts/kaggle_vllm.py --python {VLLM_PY} --preset {LLM_PRESET}", check=False, cwd=REPO)
sh("nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv", check=False)


In [ ]:
import os, sys, subprocess

WORK = "/kaggle/working"
REPO = f"{WORK}/clipping-studio"
STUDIO_PY = f"{WORK}/venv-studio/bin/python"
VLLM_PY = f"{WORK}/venv-vllm/bin/python"
LLM_PRESET = globals().get("LLM_PRESET", "qwen3-4b-awq")   # or "gemma4-e2b"


def sh(cmd, check=True, cwd=None, env=None):
    """Run a shell command, stream its output live, stop cleanly when the cell is interrupted."""
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, executable="/bin/bash",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in p.stdout:
            print(line, end="")
        p.wait()
    except KeyboardInterrupt:
        p.terminate()
        raise
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p.returncode

# ============ CELL 5 - Start FastAPI backend (port 8000) + ngrok tunnel ============
# KEEP THIS CELL RUNNING while you use the tool. The public URL is printed after the health check passes.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["NGROK_AUTHTOKEN"] = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
except Exception as exc:
    print("Could not read Kaggle secret NGROK_AUTHTOKEN:", exc)
    print("Add it via Add-ons -> Secrets, or set os.environ['NGROK_AUTHTOKEN'] manually.")

env = {**os.environ, "PYTHONPATH": REPO}
sh(f"{STUDIO_PY} scripts/kaggle_studio.py", cwd=REPO, env=env, check=False)
